# 21.11 — Case-based reasoning

Case-based reasoning solves the new problem by retrieving a similar old one, adapting it, checking it, and retaining the lesson.

CBR is symbolic memory with a similarity metric. It retrieves a precedent, adapts its solution, revises against feedback, and retains the corrected case.

Save a copy to Drive to edit.

In [ ]:

import math
import random

import matplotlib.pyplot as plt

random.seed(2111)


def weighted_distance(case_features, query_features, weights):
    total = 0.0
    for value, query, weight in zip(case_features, query_features, weights):
        total += weight * (value - query) ** 2
    return math.sqrt(total)


def build_cbr_ladder():
    rungs = []
    rungs.append({
        "name": "D1 three 2D cases",
        "cases": [
            {"id": "case1", "features": (1.0, 1.0), "solution": 7.0},
            {"id": "case2", "features": (2.0, 2.0), "solution": 10.0},
            {"id": "case3", "features": (5.0, 5.0), "solution": 20.0},
        ],
        "query": (2.2, 2.0),
        "weights": (1.0, 1.0),
        "slope": 2.0,
        "actual": 11.5,
        "expected_nearest": "case2",
    })
    rungs.append({
        "name": "D2 ten clean cases",
        "cases": [{"id": "c" + str(i), "features": (float(i), float(i % 3)), "solution": 5.0 + i} for i in range(10)],
        "query": (6.1, 0.2),
        "weights": (1.0, 1.0),
        "slope": 1.0,
        "actual": 11.8,
        "expected_nearest": "c6",
    })
    rungs.append({
        "name": "D3 metric-weight conflict and outlier",
        "cases": [
            {"id": "close_unweighted", "features": (3.0, 9.0), "solution": 30.0},
            {"id": "close_weighted", "features": (4.0, 2.0), "solution": 15.0},
            {"id": "outlier", "features": (20.0, 20.0), "solution": 100.0},
        ],
        "query": (4.1, 1.9),
        "weights": (3.0, 1.0),
        "slope": 1.5,
        "actual": 15.2,
        "expected_nearest": "close_weighted",
    })
    rungs.append({
        "name": "D4 support-ticket case library",
        "cases": [{"id": "ticket" + str(i), "features": (float(i % 5), float(i // 2), float(i % 2)), "solution": 20.0 + i} for i in range(18)],
        "query": (3.2, 6.0, 1.0),
        "weights": (2.0, 1.0, 4.0),
        "slope": 0.8,
        "actual": 33.0,
        "expected_nearest": "ticket13",
    })
    noisy_cases = []
    for i in range(35):
        noisy_cases.append({"id": "n" + str(i), "features": (float(i % 7), float((i * 2) % 9), float(i % 3), float(i % 5)), "solution": 40.0 + 0.7 * i})
    noisy_cases.append({"id": "best_sparse", "features": (5.0, 4.0, 2.0, 3.0), "solution": 62.0})
    rungs.append({
        "name": "D5 sparse noisy case base",
        "cases": noisy_cases,
        "query": (5.2, 4.1, 2.0, 3.3),
        "weights": (3.0, 1.0, 2.0, 2.0),
        "slope": 1.2,
        "actual": 62.5,
        "expected_nearest": "best_sparse",
    })
    return rungs


## The concept, built once (D1)

CBR retrieves by similarity. With $d_i=\sqrt{\sum_j w_j(x_{ij}-q_j)^2}$, query $(2.2,2.0)$ is distance $\sqrt{0.2^2+0^2}=0.2$ from case $(2,2)$. With weights $(3,1)$ the distance is $\sqrt{3\cdot0.2^2+0}=0.346$.

In [ ]:

lesson_case = (2.0, 2.0)
lesson_query = (2.2, 2.0)
lesson_distance = weighted_distance(lesson_case, lesson_query, (1.0, 1.0))
lesson_weighted_distance = weighted_distance(lesson_case, lesson_query, (3.0, 1.0))

assert round(lesson_distance, 3) == 0.2
assert round(lesson_weighted_distance, 3) == 0.346

print("plain distance", lesson_distance)
print("weighted distance", lesson_weighted_distance)


The full cycle retrieves, adapts, revises, and retains. The lesson adapts old solution $10$ by slope $2$ and feature change $1$ to get $12$, then revision error is $|12-11.5|=0.5$.

In [ ]:

def retrieve_adapt_revise_retain(cases, query, weights, slope, actual=None):
    ranked = []
    for case in cases:
        distance = weighted_distance(case["features"], query, weights)
        ranked.append((distance, case))
    ranked.sort(key=lambda item: item[0])
    nearest_distance, nearest_case = ranked[0]
    feature_change = sum(query) - sum(nearest_case["features"])
    adapted = nearest_case["solution"] + slope * feature_change
    error = None
    retained_cases = list(cases)
    if actual is not None:
        error = abs(adapted - actual)
        retained_cases.append({
            "id": "retained_" + str(len(cases)),
            "features": tuple(query),
            "solution": actual,
        })
    return {
        "nearest": nearest_case,
        "distance": nearest_distance,
        "ranked": ranked,
        "adapted": adapted,
        "error": error,
        "retained_cases": retained_cases,
        "searched": len(cases),
    }

lesson_cases = [
    {"id": "case1", "features": (1.0, 1.0), "solution": 7.0},
    {"id": "case2", "features": (2.0, 2.0), "solution": 10.0},
    {"id": "case3", "features": (5.0, 5.0), "solution": 20.0},
]
lesson_cycle = retrieve_adapt_revise_retain(lesson_cases, (2.5, 2.5), (1.0, 1.0), 2.0, actual=11.5)

assert lesson_cycle["nearest"]["id"] == "case2"
assert round(lesson_cycle["adapted"], 3) == 12.0
assert round(lesson_cycle["error"], 3) == 0.5

print("nearest", lesson_cycle["nearest"]["id"])
print("adapted", lesson_cycle["adapted"])
print("revision error", lesson_cycle["error"])


## The dataset ladder

D1–D5 grow the case library, add weights, outliers, sparse features, and noisy D5 precedent retrieval.

In [ ]:

cbr_rungs = build_cbr_ladder()

for rung in cbr_rungs:
    print(rung["name"])
    print("  cases", len(rung["cases"]))
    print("  dimensions", len(rung["query"]))
    print("  sample", rung["cases"][:2])


## Run the same method across D1–D5

In [ ]:

cbr_results = []

for rung in cbr_rungs:
    result = retrieve_adapt_revise_retain(rung["cases"], rung["query"], rung["weights"], rung["slope"], actual=rung["actual"])
    assert result["nearest"]["id"] == rung["expected_nearest"]
    cbr_results.append({
        "name": rung["name"],
        "cases": len(rung["cases"]),
        "nearest": result["nearest"]["id"],
        "distance": result["distance"],
        "error": result["error"],
        "searched": result["searched"],
        "ranked": result["ranked"],
    })

print("rung | nearest | cases searched | distance | adaptation error")
for item in cbr_results:
    print(item["name"], item["nearest"], item["searched"], round(item["distance"], 3), round(item["error"], 3))


## Results visualization

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))

for index, item in enumerate(cbr_results):
    ax = axes[0][index]
    distances = [pair[0] for pair in item["ranked"][:8]]
    ax.bar(range(len(distances)), distances)
    ax.set_title("D" + str(index + 1) + " ranks")
    ax.set_xlabel("rank")
    ax.set_ylabel("distance")

x_values = list(range(1, 6))
errors = [item["error"] for item in cbr_results]
searched = [item["searched"] for item in cbr_results]
axes[1][0].plot(x_values, errors, marker="o", label="adaptation error")
axes[1][0].plot(x_values, searched, marker="s", label="cases searched")
axes[1][0].set_xticks(x_values)
axes[1][0].set_xlabel("rung")
axes[1][0].set_ylabel("value")
axes[1][0].set_title("error/search cost vs size")
axes[1][0].legend()
for ax in axes[1][1:]:
    ax.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung

Pitfall: choosing a representation that hides the constraint via poor features. D5 needs weighted features; unweighted retrieval can pick a misleading noisy case.

In [ ]:

d5 = cbr_rungs[-1]
wrong_weights = tuple(1.0 for _ in d5["weights"])
wrong = retrieve_adapt_revise_retain(d5["cases"], d5["query"], wrong_weights, d5["slope"], actual=d5["actual"])
fixed = retrieve_adapt_revise_retain(d5["cases"], d5["query"], d5["weights"], d5["slope"], actual=d5["actual"])

print("unweighted nearest", wrong["nearest"]["id"], "error", round(wrong["error"], 3))
print("weighted nearest", fixed["nearest"]["id"], "error", round(fixed["error"], 3))
print("weighted explanation features", d5["weights"])


## Evaluate it + Practice

- Metric: retrieval correctness, adaptation absolute error, cases searched.
- No-skill baseline: reuse the first case without similarity.
- Cheap sanity check: D1 distances are 0.2 and 0.346 for the lesson case.
- Ablation: use all weights as 1 and D5 retrieval can change.
- Failure signals: nearest case lacks feature-distance explanation.

Practice 1: Change the D2 instance by one constraint and predict the metric before running.

Practice 2: Add one contradictory or noisy condition to D4 and explain how the trace changes.

Practice 3: On D5, record the first step where pruning or explanation prevents a wrong answer.